Auxiliary script to strip, clean and rename superblockify gkpg exports to geojson.

In [ ]:
import os
import glob
import shutil
import csv
import geopandas as gpd

In [ ]:
import_path = '/Users/mszell/Tresorit/bikenetkitshare/'
exports_path_gpkg = "/Users/mszell/Tresorit/bikenetkitshare/latest/superblockify_addition1/"
exports_path_geojson = "/Users/mszell/Tresorit/bikenetkitshare/latest/superblockify/"
citiesaddedpathfile = '/Users/mszell/Github/BikeNetKit/dataexport/cities/development/european_addition1.csv'

In [ ]:
with open(citiesaddedpathfile, mode='r') as infile:
    reader = csv.reader(infile, delimiter=";")
    header = next(reader)
    cities = {rows[1]: {header[0]: rows[0], header[2]: rows[2], header[3]: rows[3], header[4]: rows[4]}  for rows in reader}

## Read, clean, and write data

In [ ]:
for fp in glob.glob(exports_path_gpkg + "*.gpkg"):
    if os.path.isfile(fp):
        sb_export = {}
        sb_export["edges"] = gpd.read_file(fp, layer="edges")
        sb_export["blocks"] = gpd.read_file(fp, layer="ltns")
        sb_export["boundary"] = gpd.read_file(fp, layer="graph_meta")
        file_name = os.path.splitext(os.path.basename(fp))[0]

        try:
            # Frederiksberg_residential - name-based
            city_name = file_name.split("_")[0]
            method = file_name.split("_")[1]
            cityid = cities[city_name]["cityid"]
        except:
            # frederiksberg_dk_residential - cityid-based
            cityid = file_name.split("_")[0]+"_"+file_name.split("_")[1]
            method = file_name.split("_")[2]
        
        print(cityid, method)

        # Edges
        edges = sb_export["edges"][["classification", "edge_betweenness_linear", "edge_betweenness_linear_restricted", "population", "area", "geometry"]]
        for c in ["population", "area"]:
            edges[c] = round(edges[c], 4)
        edges.to_file(exports_path_geojson+cityid+"-superblockify-"+method+"-edges.geojson", driver="GeoJSON", RFC7946="YES")

        # Blocks
        blocks = sb_export["blocks"][["classification", "street_length_total", "street_segment_count", "population", "area", "population_density",  "geometry"]]
        for c in ["street_length_total", "street_segment_count", "population", "area"]:
            blocks[c] = blocks[c].astype(int)
        blocks.to_file(exports_path_geojson+cityid+"-superblockify-"+method+"-blocks.geojson", driver="GeoJSON", RFC7946="YES")

        # Boundary
        # Temporary hack to replace superblockify's with our city boundary
        # For cities that have only shape files like Copenhagen, this does not work!
        shutil.copyfile(import_path+"boundaries/"+cityid+".geojson", exports_path_geojson+cityid+"-superblockify-city_boundary.geojson")
        